# Crack + Spalling (YOLOv8n-seg, DPU-aware) — GTX 1650 (4 GB)

Fine-tunes the DPU-compatible YOLOv8n-seg on **two classes: `crack` and `spalling`**, validates on
both the `val` and held-out `test` splits, predicts the test split, and exports the run as a zip.

**Pipeline**

1. Set up the environment and get the repo.
2. Download `crack-01-rshs1` v17 (crack), `concrete-crack-spalling` v2 (crack + spalling) and
   `concrete-spalling-sm5v3` v1 (spalling), then merge them into one 2-class YOLO-seg dataset by
   class name.
3. Warm-start from an existing `best.pt` (the class head is rebuilt; backbone, neck and the
   class-agnostic mask branch transfer), otherwise from `yolov8n-seg-dpu-base.pt`.
4. Train, copying `last.pt`/`best.pt` to `runs_backup/<run>_live/` every `SYNC_SECONDS` seconds.
5. Validate `val` + `test`, write per-class JSON, predict, and zip everything.

## Before you run

Run this from the repo root (`ultralytics/`). It uses:

- datasets: `../datasets/` (already downloaded once; the download cell skips what exists)
- starting weights: the newest `runs/**/best.pt`, else `checkpoints/yolov8n-seg-dpu-base.pt`

Expect **~9–12 min/epoch** on the 4 GB GTX 1650 at 512 px, so 60 epochs is roughly 10–12 hours.
Interrupt any time and resume with cell 7.

## Attribution

| Resource | Version | Images | Author | License | Link |
| :--- | :---: | :---: | :--- | :--- | :--- |
| `crack-01-rshs1` | v17 | 4,977 | Crack Detection | MIT | https://universe.roboflow.com/crack-detection-slmz7/crack-01-rshs1 |
| `concrete-crack-spalling` | v2 | 1,390 | Roboflow Universe workspace `1-7frt7` | CC BY 4.0 | https://universe.roboflow.com/1-7frt7/concrete-crack-spalling |
| `concrete-spalling-sm5v3` | v1 | 999 | Roboflow Universe workspace `1-7frt7` | CC BY 4.0 | https://universe.roboflow.com/1-7frt7/concrete-spalling-sm5v3 |
| YOLOv8 / Ultralytics | — | — | G. Jocher, A. Chaurasia, J. Qiu | AGPL-3.0 | https://github.com/ultralytics/ultralytics |
| DPU-aware approximations | DSD 2026 | — | S. Karki, Q. A. Ahmed, T. Jungeblut | — | "No Attention, No Problem", doi:10.23919/DSDCPS00089.2026.00025 |

> **Keep the CC BY 4.0 attribution** if you publish a model or a merged dataset built here.
> **Do not commit your Roboflow API key.** These notebooks ask for it at runtime.

In [ ]:
#@title 1) Environment and repository

import subprocess
import sys
from pathlib import Path

REPO_DIR = Path.cwd()
if not (REPO_DIR / "ultralytics").is_dir():
    REPO_DIR = Path("/home/vicosilalahi/programming/yolo-fpga/ultralytics")
assert (REPO_DIR / "ultralytics").is_dir(), f"Start this notebook from the repo root, not {REPO_DIR}"

try:
    import ultralytics
except ModuleNotFoundError:
    print("ultralytics is not importable from", sys.executable)
    print("Install it once with:")
    print(f"  {sys.executable} -m pip install -e {REPO_DIR}")
    raise

import torch

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
#@title 2) Configuration { display-mode: "form" }

from pathlib import Path

import torch

REPO_DIR = Path.cwd()
if not (REPO_DIR / "ultralytics").is_dir():
    REPO_DIR = Path("/home/vicosilalahi/programming/yolo-fpga/ultralytics")
DATA_ROOT = REPO_DIR.parent / "datasets"
MERGED_DIR = DATA_ROOT / "concrete-2cls"
ARTIFACT_DIR = REPO_DIR                    # runs_backup/, exports/ and checkpoints/ live here
RUNS_BACKUP = ARTIFACT_DIR / "runs_backup"
EXPORT_DIR = ARTIFACT_DIR / "exports"

EPOCHS = 60
BATCH = 4                                   # 4 GB VRAM
IMGSZ = 512                                 # 640 fits but is slower; 512 is a good trade
LR0 = 0.001
WORKERS = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
VAL_DEVICE = "0" if torch.cuda.is_available() else "cpu"
SYNC_SECONDS = 60                          # local disk: a minute of work is cheap to lose
RUN_NAME = "concrete-2cls"
WARM_START = "auto"   # "auto" | "best" | "base"
WEIGHTS_PATH = None   # e.g. "/home/you/runs/best.pt"
EXTRA_WEIGHT_DIRS = []
WEIGHTS_HINT = "Put best.pt under the repo (runs/ and checkpoints/ are scanned), or set WEIGHTS_PATH."

# crack-01-rshs1: the dataset runs/best.pt was trained on (v17 = 4,977 images, 1 class).
CRACK_WORKSPACE, CRACK_PROJECT, CRACK_VERSION = "crack-detection-slmz7", "crack-01-rshs1", 17
# concrete-crack-spalling v2 = 1,390 images, no augmentation, train 1,042 / valid 348 / no test.
SPALLING_WORKSPACE, SPALLING_PROJECT, SPALLING_VERSION = "1-7frt7", "concrete-crack-spalling", 2
# Optional third source: spalling-only images, useful because spalling is the thin class.
EXTRA_SPALLING = True
EXTRA_WORKSPACE, EXTRA_PROJECT, EXTRA_VERSION = "1-7frt7", "concrete-spalling-sm5v3", 1

GLOBAL_NAMES = ["crack", "spalling"]   # unified class ids: 0 and 1
TEST_FRACTION_FROM_VAL = 1 / 3         # sources without a test split: carve this share of val

print("Config OK | warm start:", WARM_START, "| epochs:", EPOCHS, "| batch:", BATCH, "| device:", DEVICE)

In [ ]:
#@title 3) Download the Roboflow datasets (skipped when already present)

import shutil
from pathlib import Path

CRACK_DIR = DATA_ROOT / "crack-01"
SPALL_DIR = DATA_ROOT / "concrete-crack-spalling"
EXTRA_DIR = DATA_ROOT / "concrete-spalling-sm5v3" if EXTRA_SPALLING else None


def _ready(path):
    return path is not None and (path / "data.yaml").exists()


def download_dataset(rf, workspace, project, version, dest):
    proj = rf.workspace(workspace).project(project)
    versions = proj.versions()
    print(f"\n{workspace}/{project}: {len(versions)} versions")
    for v in versions:
        print(f"  v{v.version}: created={v.created} images={v.images} splits={v.splits}")
    if version is None:
        target = max(versions, key=lambda v: str(v.created))
    else:
        target = next(v for v in versions if int(v.version) == int(version))
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    ds = target.download("yolov8", location=str(dest), overwrite=True)
    print(f"-> using v{target.version} at {ds.location}")
    return Path(ds.location)


if any(not _ready(p) for p in (CRACK_DIR, SPALL_DIR, EXTRA_DIR) if p is not None):
    import os
    from getpass import getpass

    from roboflow import Roboflow

    API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
    rf = Roboflow(api_key=API_KEY)
    if not _ready(CRACK_DIR):
        CRACK_DIR = download_dataset(rf, CRACK_WORKSPACE, CRACK_PROJECT, CRACK_VERSION, CRACK_DIR)
    if not _ready(SPALL_DIR):
        SPALL_DIR = download_dataset(rf, SPALLING_WORKSPACE, SPALLING_PROJECT, SPALLING_VERSION, SPALL_DIR)
    if EXTRA_DIR is not None and not _ready(EXTRA_DIR):
        EXTRA_DIR = download_dataset(rf, EXTRA_WORKSPACE, EXTRA_PROJECT, EXTRA_VERSION, EXTRA_DIR)
else:
    print("Using the datasets already present under", DATA_ROOT)

print("\ncrack:", CRACK_DIR)
print("spalling:", SPALL_DIR)
print("extra:", EXTRA_DIR)

In [ ]:
#@title 4) Merge into one 2-class dataset (name-based remap)
import shutil
import zlib
from collections import Counter

import yaml

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def _names(cfg):
    names = cfg.get("names")
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda x: int(x))]
    return [str(n) for n in names]


def _split_dir(base, entry):
    # Resolve a data.yaml split entry, or return None when the directory is absent.
    p = Path(str(entry))
    if p.is_absolute():
        return p if p.exists() else None
    for cand in ((base / p), (base / str(p).replace("../", ""))):
        cand = cand.resolve()
        if cand.exists():
            return cand
    return None


def merge_dataset(ds_dir, prefix, global_names, merged, stats):
    ds_dir = Path(ds_dir)
    cfg = _yaml(ds_dir / "data.yaml")
    src_names = _names(cfg)
    mapping = {}
    for i, name in enumerate(src_names):
        key = name.strip().lower()
        if key in global_names:
            mapping[i] = global_names.index(key)
        else:
            mapping[i] = None
            print(f"  WARNING: {prefix} class {name!r} is not used here; its annotations are dropped")
    print(f"{prefix}: classes {src_names} -> {mapping}")

    base = Path(cfg.get("path", ds_dir))
    if not base.is_absolute():
        base = (ds_dir / base).resolve()

    # Roboflow yamls often declare test: ../test/images without shipping that directory.
    src_has_test = _split_dir(base, cfg.get("test")) is not None
    if not src_has_test:
        print(f"  {prefix} ships no test split; carving ~{TEST_FRACTION_FROM_VAL:.0%} of its val split into test")

    for split in ("train", "val", "test"):
        out_img = merged / split / "images"
        out_lbl = merged / split / "labels"
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)

    for split in ("train", "val", "test"):
        entry = cfg.get(split) or (cfg.get("valid") if split == "val" else None)
        img_dir = _split_dir(base, entry) if entry else None
        if img_dir is None:
            print(f"  {split}: no directory, skipped")
            continue
        lbl_dir = img_dir.parent / "labels"

        n_img = n_lbl = 0
        routed = Counter()
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            out_split = split
            if split == "val" and not src_has_test:
                if (zlib.crc32(img.stem.encode()) / 0xFFFFFFFF) < TEST_FRACTION_FROM_VAL:
                    out_split = "test"
            stem = f"{prefix}__{img.stem}"
            shutil.copy2(img, merged / out_split / "images" / (stem + img.suffix))
            lines = []
            src_lbl = lbl_dir / f"{img.stem}.txt"
            if src_lbl.exists():
                for line in src_lbl.read_text().splitlines():
                    parts = line.split()
                    if not parts:
                        continue
                    cid = int(float(parts[0]))
                    target = mapping.get(cid)
                    if target is None:
                        stats["dropped_instances"] += 1
                        continue
                    coords = [float(v) for v in parts[1:]]
                    if len(coords) == 4:  # detection label: bbox -> rectangle polygon
                        cx, cy, w, h = coords
                        x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
                        coords = [x1, y1, x2, y1, x2, y2, x1, y2]
                        stats["boxes_as_rectangles"] += 1
                    elif len(coords) < 6 or len(coords) % 2:
                        continue
                    lines.append(" ".join([str(target)] + [f"{v:.6f}" for v in coords]))
                    stats["instances"][out_split][global_names[target]] += 1
                    n_lbl += 1
            (merged / out_split / "labels" / f"{stem}.txt").write_text(
                ("\n".join(lines) + "\n") if lines else ""
            )
            stats["images"][out_split] += 1
            routed[out_split] += 1
            n_img += 1
        carved = f" ({routed['test']} of them routed to test)" if split == "val" and routed["test"] else ""
        print(f"  {split}: {n_img} images, {n_lbl} instances{carved}")


stats = {
    "images": Counter(),
    "instances": {s: Counter() for s in ("train", "val", "test")},
    "boxes_as_rectangles": 0,
    "dropped_instances": 0,
}

if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
MERGED_DIR.mkdir(parents=True)
merge_dataset(CRACK_DIR, "crack01", GLOBAL_NAMES, MERGED_DIR, stats)
merge_dataset(SPALL_DIR, "spall", GLOBAL_NAMES, MERGED_DIR, stats)
if EXTRA_DIR is not None:
    merge_dataset(EXTRA_DIR, "spall2", GLOBAL_NAMES, MERGED_DIR, stats)

merged_cfg = {
    "path": str(MERGED_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(GLOBAL_NAMES),
    "names": {i: n for i, n in enumerate(GLOBAL_NAMES)},
}
with open(MERGED_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(merged_cfg, f, sort_keys=False)

print("\nMerged dataset summary")
print(f"{'split':<8}{'images':>8}{'crack':>8}{'spalling':>10}")
for split in ("train", "val", "test"):
    print(f"{split:<8}{stats['images'][split]:>8}{stats['instances'][split]['crack']:>8}{stats['instances'][split]['spalling']:>10}")
if stats["boxes_as_rectangles"]:
    print(f"\nWARNING: {stats['boxes_as_rectangles']} detection labels were converted to rectangle masks.")
    print("Prefer an instance-segmentation export; rectangle masks hurt spalling quality.")
if stats["dropped_instances"]:
    print(f"\n{stats['dropped_instances']} annotations of unused classes were dropped.")
print("\ndata.yaml:", MERGED_DIR / "data.yaml")

In [ ]:
#@title 5) Pick the starting weights { display-mode: "form" }

import glob
import shutil

import torch

CKPT_DIR = ARTIFACT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def _usable(path):
    # True if this checkpoint is a crack/spalling model we can transfer from.
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        names = getattr(ckpt.get("model"), "names", None) or {}
        return 1 <= len(names) <= 2 and all(n in ("crack", "spalling") for n in names.values())
    except Exception:
        return False


# Priority: explicit WEIGHTS_PATH, then any *best*.pt in the scanned dirs, then the DPU base.
scanned = [ARTIFACT_DIR] + [Path(d) for d in EXTRA_WEIGHT_DIRS if Path(d).is_dir()]

if WEIGHTS_PATH:
    src = Path(WEIGHTS_PATH)
    if not src.is_file():
        raise FileNotFoundError(f"WEIGHTS_PATH={WEIGHTS_PATH!r} does not exist. {WEIGHTS_HINT}")
    best_candidates = [src]
else:
    best_candidates = []
    for d in scanned:
        best_candidates += [Path(p) for p in glob.glob(str(d / "**" / "*best*.pt"), recursive=True)]
    best_candidates = [p for p in best_candidates if p.is_file()]

# A COCO or unrelated best.pt is worse than no warm start: keep only crack/spalling models.
usable = [p for p in best_candidates if _usable(p)]
for skipped in [p for p in best_candidates if p not in usable]:
    print("skipping (not a crack/spalling model):", skipped)

BASE_CANDIDATES = [
    ARTIFACT_DIR / "yolov8n-seg-dpu-base.pt",
    REPO_DIR / "checkpoints" / "yolov8n-seg-dpu-base.pt",
]

if WARM_START == "best" and not usable:
    raise FileNotFoundError(
        f"WARM_START='best' but no crack/spalling *best*.pt was found in "
        f"{[str(d) for d in scanned]}.\n"
        f"  {WEIGHTS_HINT}"
    )

if WARM_START in ("auto", "best") and usable:
    src = max(usable, key=lambda p: p.stat().st_mtime)
    weights = CKPT_DIR / "warm-start.pt"
    shutil.copy2(src, weights)
    print("Warm start from:", src)
    print("The class head is rebuilt; backbone, neck and the mask branch transfer.")
else:
    base_src = next((p for p in BASE_CANDIDATES if p.exists()), None)
    if base_src is None:
        raise FileNotFoundError(
            "No starting weights found.\n"
            f"  {WEIGHTS_HINT}\n"
            f"  scanned: {[str(d) for d in scanned]}"
        )
    weights = base_src
    print("Warm start from the DPU base:", weights)

print("Weights:", weights)

In [ ]:
#@title 6) Train the 2-class model { display-mode: "form" }

import shutil
import threading
import time

from ultralytics import YOLO

RUNS_BACKUP.mkdir(parents=True, exist_ok=True)
LIVE_DIR = RUNS_BACKUP / f"{RUN_NAME}_live"
LIVE_DIR.mkdir(parents=True, exist_ok=True)
for stale in LIVE_DIR.glob("*.tmp"):
    stale.unlink(missing_ok=True)  # a killed session can leave a half-copied file behind

# Ultralytics writes here; the name is deterministic because exist_ok=True.
RUN_DIR = REPO_DIR / "runs/colab" / RUN_NAME

_stop = threading.Event()


def _copy(src, dst, min_age=2.0):
    # Skip a file written in the last couple of seconds: torch.save may still be writing it.
    if not src.exists():
        return False
    size = src.stat().st_size
    if size == 0 or (time.time() - src.stat().st_mtime) < min_age:
        return False
    tmp = dst.with_name(dst.name + ".tmp")
    shutil.copy2(src, tmp)
    if tmp.stat().st_size != size:  # grew while copying: torn read, retry on the next round
        tmp.unlink(missing_ok=True)
        return False
    tmp.replace(dst)  # atomic: a kill can leave a .tmp, never a truncated checkpoint
    return True


def _sync_once(min_age=2.0):
    for name in ("last.pt", "best.pt"):
        _copy(RUN_DIR / "weights" / name, LIVE_DIR / name, min_age=min_age)
    _copy(RUN_DIR / "results.csv", LIVE_DIR / "results.csv", min_age=min_age)


def _sync_loop():
    # A thread survives DDP, unlike a per-epoch callback registered on the model.
    while not _stop.wait(SYNC_SECONDS):
        _sync_once()


sync_thread = threading.Thread(target=_sync_loop, daemon=True)
sync_thread.start()

try:
    model = YOLO(str(weights))
    model.train(
        data=str(MERGED_DIR / "data.yaml"),
        epochs=EPOCHS,
        batch=BATCH,
        imgsz=IMGSZ,
        optimizer="AdamW",
        lr0=LR0,
        lrf=0.01,
        patience=30,
        device=DEVICE,
        workers=WORKERS,
        amp=True,
        plots=True,
        seed=0,
        project=str(REPO_DIR / "runs/colab"),
        name=RUN_NAME,
        exist_ok=True,
    )
finally:
    _stop.set()
    _sync_once(min_age=0)  # training is over, nothing is being written anymore

BEST = RUN_DIR / "weights/best.pt"
print("\nRun directory:", RUN_DIR)
print("Best weights: ", BEST)
print("Live backup:  ", LIVE_DIR)

# Full copy (plots, csv, args) once training has settled.
full = RUNS_BACKUP / f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M')}"
if RUN_DIR.exists():
    if full.exists():
        shutil.rmtree(full)
    shutil.copytree(RUN_DIR, full)
    print("Full run backup:", full)

In [ ]:
#@title 7) Resume an interrupted run

from pathlib import Path

import torch
from ultralytics import YOLO

LIVE = RUNS_BACKUP / f"{RUN_NAME}_live" / "last.pt"
if not LIVE.exists():
    print("No live checkpoint at", LIVE)
    print("Run the training cell first, or point RUN_NAME at the interrupted run.")
else:
    ckpt = torch.load(LIVE, map_location="cpu", weights_only=False)
    resumable = ckpt.get("epoch", -1) >= 0 and ckpt.get("optimizer") is not None
    if not resumable:
        # A finished run has its optimizer stripped; resume=True would silently start a new
        # training run on the task's default dataset (COCO), which is never what you want.
        print("This checkpoint is a finished/stripped model, not an interrupted run.")
        print("Use it as weights instead:  model = YOLO(str(LIVE)); model.train(data=..., ...)")
    else:
        print(f"Resuming from epoch {ckpt['epoch']} ...")
        model = YOLO(str(LIVE))
        model.train(resume=True)
        RUN_DIR = Path(model.trainer.save_dir)
        BEST = RUN_DIR / "weights/best.pt"
        print("\nResumed. Run directory:", RUN_DIR)
        print("Best weights: ", BEST)

In [ ]:
#@title 8) Validate on val and test (FP32)

import subprocess
import sys

METRICS_DIR = REPO_DIR / "metrics"
METRICS_DIR.mkdir(exist_ok=True)


def run_validate(split, name, out_json):
    cmd = [
        sys.executable, str(REPO_DIR / "validate.py"),
        "--weights", str(BEST),
        "--data", str(MERGED_DIR / "data.yaml"),
        "--split", split,
        "--imgsz", str(IMGSZ),
        "--batch", str(BATCH),
        "--device", str(VAL_DEVICE),
        "--full-precision",
        "--project", str(REPO_DIR / "runs/val"),
        "--name", name,
        "--output-json", str(out_json),
    ]
    print("\n$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, check=True)


run_validate("val", f"{RUN_NAME}_val", METRICS_DIR / "results_concrete_2cls_val.json")
run_validate("test", f"{RUN_NAME}_test", METRICS_DIR / "results_concrete_2cls_test.json")

for p in sorted(METRICS_DIR.glob("results_concrete_2cls_*.json")):
    print(f"\n{p.name}:\n{p.read_text()}")

In [ ]:
#@title 9) Per-class metrics (JSON)
import json

from ultralytics import YOLO

eval_model = YOLO(str(BEST))
m = eval_model.val(
    data=str(MERGED_DIR / "data.yaml"),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=False,
    verbose=False,
    project=str(REPO_DIR / "runs/val"),
    name=f"{RUN_NAME}_per_class",
    exist_ok=True,
)
names = getattr(m, "names", eval_model.names)

print(f"{'class':<12}{'box P':>8}{'box R':>8}{'box mAP50':>11}{'box mAP50-95':>14}"
      f"{'mask P':>9}{'mask R':>9}{'mask mAP50':>12}{'mask mAP50-95':>15}")
per_class = {}
for i, name in names.items():
    per_class[name] = {
        "box": {
            "precision": float(m.box.p[i]),
            "recall": float(m.box.r[i]),
            "map50": float(m.box.ap50[i]),
            "map50_95": float(m.box.ap[i]),
        },
        "mask": {
            "precision": float(m.seg.p[i]),
            "recall": float(m.seg.r[i]),
            "map50": float(m.seg.ap50[i]),
            "map50_95": float(m.seg.ap[i]),
        },
    }
    print(f"{name:<12}{m.box.p[i]:>8.3f}{m.box.r[i]:>8.3f}{m.box.ap50[i]:>11.3f}{m.box.ap[i]:>14.3f}"
          f"{m.seg.p[i]:>9.3f}{m.seg.r[i]:>9.3f}{m.seg.ap50[i]:>12.3f}{m.seg.ap[i]:>15.3f}")

out = METRICS_DIR / "results_concrete_2cls_test_per_class.json"
with open(out, "w") as f:
    json.dump(per_class, f, indent=2)
print("\nSaved", out)

# Incremental backup: keep metrics in Drive even if the runtime dies before the zip cell.
import shutil

DRIVE_METRICS = ARTIFACT_DIR / "metrics"
DRIVE_METRICS.mkdir(parents=True, exist_ok=True)
for p in sorted(METRICS_DIR.glob("results_concrete_2cls_*.json")):
    shutil.copy2(p, DRIVE_METRICS / p.name)
print("Metrics copied to", DRIVE_METRICS)

In [ ]:
#@title 10) Predict the test split
from IPython.display import Image as IPImage
from IPython.display import display

results = eval_model.predict(
    source=str(MERGED_DIR / "test/images"),
    imgsz=IMGSZ,
    conf=0.25,
    max_det=100,
    save=True,
    project=str(REPO_DIR / "runs/predict"),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)
PRED_DIR = Path(results[0].save_dir)
print("Predictions saved to:", PRED_DIR)
for p in sorted(PRED_DIR.glob("*.jpg"))[:3]:
    print(p.name)
    display(IPImage(filename=str(p), width=640))

import shutil

drive_preds = ARTIFACT_DIR / "predictions" / f"{RUN_NAME}_test"
if drive_preds.exists():
    shutil.rmtree(drive_preds)
shutil.copytree(PRED_DIR, drive_preds)
print("Predictions copied to", drive_preds)

In [ ]:
#@title 11) Write RESULTS.md and export a zip
import json
import time
import zipfile

METRICS_DIR = REPO_DIR / "metrics"
val = json.loads((METRICS_DIR / "results_concrete_2cls_val.json").read_text())
test = json.loads((METRICS_DIR / "results_concrete_2cls_test.json").read_text())


def pct(x):
    return f"{100 * x:.2f}%"


results_md = f'''# Crack + spalling run: {RUN_NAME}

Merged dataset: `crack-01-rshs1` v17 (crack) + `concrete-crack-spalling` v2 (crack, spalling)
+ `concrete-spalling-sm5v3` v1 (spalling), classes `crack`, `spalling`.
Sources without a test split had ~{TEST_FRACTION_FROM_VAL:.0%} of their val images carved into the test split.
Warm start: `{weights}`.
Epochs: {EPOCHS} | batch: {BATCH} | imgsz: {IMGSZ} | lr0: {LR0}

| Split | Box mAP50-95 | Box mAP50 | Mask mAP50-95 | Mask mAP50 |
| :--- | :---: | :---: | :---: | :---: |
| val | {pct(val['box']['map50_95'])} | {pct(val['box']['map50'])} | {pct(val['seg']['map50_95'])} | {pct(val['seg']['map50'])} |
| test | {pct(test['box']['map50_95'])} | {pct(test['box']['map50'])} | {pct(test['seg']['map50_95'])} | {pct(test['seg']['map50'])} |

Per-class metrics: `metrics/results_concrete_2cls_test_per_class.json`.

## Attribution
- `crack-01-rshs1` by Crack Detection, MIT license:
  https://universe.roboflow.com/crack-detection-slmz7/crack-01-rshs1
- `concrete-crack-spalling` by Roboflow Universe workspace `1-7frt7`, CC BY 4.0:
  https://universe.roboflow.com/1-7frt7/concrete-crack-spalling
- `concrete-spalling-sm5v3` by Roboflow Universe workspace `1-7frt7`, CC BY 4.0:
  https://universe.roboflow.com/1-7frt7/concrete-spalling-sm5v3
- Model architecture: Ultralytics YOLOv8, G. Jocher, A. Chaurasia, J. Qiu (AGPL-3.0).
- DPU-aware approximations: S. Karki, Q. A. Ahmed, T. Jungeblut,
  "No Attention, No Problem: DPU-Aware Attention Approximation in Modern YOLO on FPGA",
  DSD 2026, doi:10.23919/DSDCPS00089.2026.00025.
'''
RESULTS_PATH = REPO_DIR / "RESULTS.md"
RESULTS_PATH.write_text(results_md)
print(results_md)

stamp = time.strftime("%Y%m%d_%H%M")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
zip_path = EXPORT_DIR / f"crack_spalling_{RUN_NAME}_{stamp}.zip"


def add(z, path, arc):
    path = Path(path)
    if path.is_file():
        z.write(path, f"{arc}/{path.name}")
    elif path.is_dir():
        for f in path.rglob("*"):
            if f.is_file():
                z.write(f, f"{arc}/{f.relative_to(path)}")


with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    add(z, RUN_DIR, f"run_{RUN_NAME}")
    add(z, METRICS_DIR, "metrics")
    add(z, PRED_DIR, "predictions")
    add(z, MERGED_DIR / "data.yaml", "dataset")
    z.write(RESULTS_PATH, "RESULTS.md")

print(f"\nZip: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

print("\nLocal run finished. The zip and the run directory are on this machine:")
print("  ", zip_path)

## Notes

- **Test split:** `concrete-crack-spalling` and `concrete-spalling-sm5v3` ship no test split, so
  cell 4 moves a deterministic third of their `val` images into `test`.
- **Versions:** `concrete-crack-spalling` is pinned to **v2** (1,390 unaugmented images). v5 is
  augmented to 3,561 and v4 puts everything in `valid`, so "newest" is the wrong choice.
- **Class balance:** ~6.9k `crack` vs ~1.1k `spalling` instances in train, so `spalling` is the thin
  class. Watch its per-class AP.
- **`lr0` is respected** here because the notebook calls `model.train(optimizer="AdamW", lr0=...)`
  directly; the repo's `train.py` uses `optimizer=auto`, which ignores `--lr0`.
- **INT8:** this trains and validates FP32 only. Quantize afterwards with `dpu_workflow/`, calibrate
  with images containing **both** classes, then re-validate.
- **License:** the spalling datasets are CC BY 4.0 — keep the attribution in `RESULTS.md` and in
  any published model card or dataset.

- **No session limit:** you can close the browser; keep the machine awake. Ctrl-C stops safely —
  `last.pt` is synced to `runs_backup/<run>_live/` every minute (`SYNC_SECONDS = 60`).
- **Resume:** run cell 7 instead of cell 6. It continues from the synced checkpoint with the same
  scheduler and epoch counter.
- **VRAM:** if you hit OOM, drop `BATCH` to 2 or `IMGSZ` to 448. 4 GB is the constraint, not the model.
- **Speed:** `IMGSZ = 512` is ~40% faster than 640 and costs a little thin-crack detail.